# Lesson 23 — Decorators & Context Managers

Wrap functions with extra behaviour using decorators, and manage resources cleanly with the with statement.

## 🎯 Learning Objectives

- Understand closures (functions inside functions)
- Write decorators with and without arguments
- Preserve function metadata with `functools.wraps`
- Build context managers with classes and `contextlib`

## 🔒 Closures

An inner function that **remembers** variables from the outer function — even after the outer function has finished.

In [1]:
def make_multiplier(n):
    def multiply(x):
        return x * n          # remembers n
    return multiply

double = make_multiplier(2)
triple = make_multiplier(3)
print(double(10), triple(10))

20 30


## 🎀 Decorators

A **decorator** is a function that takes a function and returns a new function with extra behaviour.
`@decorator` above a function is shorthand for `func = decorator(func)`.

In [2]:
def announce(func):
    def wrapper():
        print("▶ Before the function runs")
        func()
        print("■ After the function runs")
    return wrapper

@announce
def say_hello():
    print("Hello!")

say_hello()

▶ Before the function runs
Hello!
■ After the function runs


### Decorators for functions with arguments + `functools.wraps`

In [3]:
import functools

def logger(func):
    @functools.wraps(func)                 # keeps the original name & docstring
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}{args} {kwargs or ''}")
        result = func(*args, **kwargs)
        print(f"{func.__name__} returned {result}")
        return result
    return wrapper

@logger
def add(a, b):
    """Add two numbers."""
    return a + b

add(3, 4)
print("Name kept:", add.__name__, "| Doc kept:", add.__doc__)

Calling add(3, 4) 
add returned 7
Name kept: add | Doc kept: Add two numbers.


### Practical decorator — timing

In [4]:
import time, functools

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {'less than 1 second' if elapsed < 1 else f'{elapsed:.2f}s'}")
        return result
    return wrapper

@timer
def slow_sum(n):
    return sum(range(n))

print(slow_sum(1_000_000))

slow_sum took less than 1 second
499999500000


### Decorators with arguments

In [5]:
import functools

def repeat(times):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(times):
                func(*args, **kwargs)
        return wrapper
    return decorator

@repeat(times=3)
def cheer(name):
    print(f"Go {name}! 🎉")

cheer("Python")

Go Python! 🎉
Go Python! 🎉
Go Python! 🎉


### Stacking decorators & a validation example

In [6]:
import functools

def require_positive(func):
    @functools.wraps(func)
    def wrapper(*args):
        if any(a <= 0 for a in args):
            return "❌ All arguments must be positive"
        return func(*args)
    return wrapper

def uppercase_result(func):
    @functools.wraps(func)
    def wrapper(*args):
        return str(func(*args)).upper()
    return wrapper

@uppercase_result
@require_positive
def area_label(w, h):
    return f"area is {w * h} sq units"

print(area_label(4, 5))
print(area_label(-1, 5))

AREA IS 20 SQ UNITS
❌ ALL ARGUMENTS MUST BE POSITIVE


### Built-in decorators you have already met

`@property`, `@classmethod`, `@staticmethod`, `@dataclass`, `@functools.lru_cache`, `@abstractmethod`.

## 🚪 Context Managers — the `with` statement

A context manager sets something up and **guarantees** clean-up, even if an error happens.
You've used one already: `with open(...) as f:`.

### Class-based (`__enter__` / `__exit__`)

In [7]:
class Timer:
    def __enter__(self):
        import time
        self.start = time.perf_counter()
        print("⏱️ Timer started")
        return self

    def __exit__(self, exc_type, exc, tb):
        import time
        self.elapsed = time.perf_counter() - self.start
        print("⏱️ Timer stopped")
        return False        # don't swallow exceptions

with Timer() as t:
    total = sum(range(100_000))
print("Total:", total, "| measured:", t.elapsed > 0)

⏱️ Timer started
⏱️ Timer stopped
Total: 4999950000 | measured: True


### Generator-based with `@contextmanager`

In [8]:
from contextlib import contextmanager

@contextmanager
def tag(name):
    print(f"<{name}>")
    try:
        yield
    finally:
        print(f"</{name}>")

with tag("html"):
    with tag("body"):
        print("  Hello, context managers!")

<html>
<body>
  Hello, context managers!
</body>
</html>


In [9]:
from contextlib import contextmanager

@contextmanager
def safe_block(label):
    try:
        yield
    except Exception as e:
        print(f"[{label}] handled {type(e).__name__}: {e}")

with safe_block("division"):
    print(10 / 0)
print("Program keeps running ✅")

[division] handled ZeroDivisionError: division by zero
Program keeps running ✅


In [10]:
from contextlib import suppress
import os
with suppress(FileNotFoundError):
    os.remove("file_that_does_not_exist.txt")
print("No crash — the error was suppressed")

No crash — the error was suppressed


## 📌 Summary

- Closures remember variables from their enclosing scope.
- Decorators wrap functions: `@decorator`; use `functools.wraps`.
- A decorator with arguments needs an extra outer function.
- Context managers (`with`) guarantee setup/clean-up via `__enter__`/`__exit__` or `@contextmanager`.